# 01: Kaggle SFT Training

This notebook runs on **GPU (T4)** for:
- Downloading datasets from HF Hub
- Training student model with LoRA
- Evaluating on test set

**Persistence**: Checkpoints saved to HF Hub.

## Setup

In [ ]:
# Install dependencies
!pip install -r requirements-train.txt

In [ ]:
# HF Hub setup
from huggingface_hub import snapshot_download, upload_folder, create_repo

# Create repos if they don't exist
try:
    create_repo("Arshia-HZ/causal-mas-distill-data", repo_type="dataset", exist_ok=True)
except Exception:
    pass

try:
    create_repo("Arshia-HZ/causal-mas-distill-ckpt", repo_type="model", exist_ok=True)
except Exception:
    pass

# Download data from HF Hub
snapshot_download(
    "Arshia-HZ/causal-mas-distill-data",
    repo_type="dataset",
    local_dir="data"
)

## Training Configuration

In [ ]:
# Import modules
from pathlib import Path
from src.distill.sft import DistillationTrainer
import json

## Train Causal Selection Model

In [ ]:
# Load causal dataset
with open("data/datasets/causal_dataset.json") as f:
    causal_examples = json.load(f)

print(f"Loaded {len(causal_examples)} causal examples")

In [ ]:
# Create trainer
trainer = DistillationTrainer(
    model_name_or_path="Qwen/Qwen2.5-1.5B-Instruct",
    output_dir=Path("checkpoints/causal"),
    training_args={
        "num_train_epochs": 3,
        "per_device_train_batch_size": 4,
        "gradient_accumulation_steps": 2,
        "learning_rate": 2e-4,
        "bf16": True,
        "save_strategy": "epoch",
    }
)

# Prepare dataset
dataset = trainer.prepare_dataset(causal_examples)

# Train
trainer.train(train_dataset=dataset, max_seq_length=4096)

## Train Baseline Models (for comparison)

In [ ]:
# List of baseline datasets to train
baselines = ["random_lenmatched", "last_round_only", "confidence", "oracle_filter"]

for baseline in baselines:
    print(f"\n{'='*50}")
    print(f"Training {baseline} baseline")
    print(f"{'='*50}")
    
    # Load dataset
    with open(f"data/datasets/{baseline}_dataset.json") as f:
        examples = json.load(f)
    
    # Create trainer
    trainer = DistillationTrainer(
        model_name_or_path="Qwen/Qwen2.5-1.5B-Instruct",
        output_dir=Path(f"checkpoints/{baseline}"),
        training_args={
            "num_train_epochs": 3,
            "per_device_train_batch_size": 4,
            "learning_rate": 2e-4,
            "bf16": True,
        }
    )
    
    # Train
    dataset = trainer.prepare_dataset(examples)
    trainer.train(train_dataset=dataset)

## Evaluation

In [ ]:
# Load test data
with open("data/test.json") as f:
    test_data = json.load(f)

print(f"Loaded {len(test_data)} test examples")

In [ ]:
from eval.run_eval import run_evaluation
import pandas as pd

results = []
models = ["causal", "random_lenmatched", "last_round_only", "confidence", "oracle_filter"]

for model in models:
    print(f"\nEvaluating {model}...")
    try:
        metrics = run_evaluation(
            model_path=f"checkpoints/{model}/final",
            test_data=test_data,
            output_dir=Path(f"eval_results/{model}"),
        )
        results.append({
            "model": model,
            "accuracy": metrics["accuracy"],
        })
    except Exception as e:
        print(f"Error: {e}")
        results.append({
            "model": model,
            "accuracy": None,
        })

# Display results
df = pd.DataFrame(results)
print("\n" + df.to_string(index=False))

## Upload Checkpoints to HF Hub

In [ ]:
# Upload all checkpoints
for model in models:
    print(f"Uploading {model}...")
    try:
        upload_folder(
            folder_path=f"checkpoints/{model}/final",
            repo_id="Arshia-HZ/causal-mas-distill-ckpt",
            repo_type="model",
            path_in_repo=f"{model}"
        )
    except Exception as e:
        print(f"Error uploading {model}: {e}")

print("Done!")